# Apt 305, 50 Barry St — pyBuildingEnergy process run

A 20 m² Melbourne apartment: one exposed (west) facade, five conditioned
neighbours, zeroed thermal mass, ideal loads.

This notebook runs the four steps **in order**:

| Step | What | Script |
| --- | --- | --- |
| **1** | Unmodified pyBuildingEnergy vs **EnergyPlus 24.1**, plus the ISO **Sankey** | `baseline_vs_energyplus.py` |
| **2–3** | Baseline → **+ window** → **+ window + thermal transmittance**, one change at a time | `compare_branches_apt305.py` |
| **4** | Baseline → **+ ventilation** → **+ latent heat** → **both** | `compare_ventilation_latent.py` |

### One weather file, everywhere

Every run below is pinned to the **same** EPW, committed in `weather_cache/`:
`AUS_VIC_Melbourne.RO.948680_TMYx.2011-2025` — Melbourne Regional Office, 0.008°
from the building. Nothing is downloaded, so no run can quietly fall back to a
different source.

> This matters. Two charts once disagreed by 4× on heating — same engine, same
> building, *different weather*: one run had fallen back to a PVGIS TMY, the
> other used an EPW. Every run now writes `run_meta.json` recording the file it
> actually used, and the last cell checks that **before** comparing numbers.
>
> An earlier revision of this repo bundled `AUS_VIC_Charlton` instead, which sits
> 2.0° inland — colder winters, hotter summers. Any figure quoted from a run on
> Charlton is not Melbourne. Cell 3 prints the station so you can see which you
> have.

## Setup

> **Private repo?** If the clone asks for credentials, create a GitHub personal
> access token with `repo` scope and use the commented `REPO` line.

In [ ]:
# 1. Clone the repository.
#
# Clone `main`: it is the harness branch — it carries all four example scripts,
# the notebook, and the correct Melbourne EPW. The engine branches are checked
# out into throwaway git worktrees by the comparison scripts themselves, so they
# must all be fetched, but none of them is what you work from.
REPO   = "https://github.com/samiraghafarigousheh-sys/AIB.git"
BRANCH = "main"

# Private repo? Uncomment and paste a token with `repo` scope:
# TOKEN = "ghp_xxx"
# REPO = f"https://{TOKEN}@github.com/samiraghafarigousheh-sys/AIB.git"

import os, shutil
if os.path.isdir("AIB"):
    shutil.rmtree("AIB")

!git clone --quiet --branch $BRANCH $REPO AIB
%cd AIB

# Every engine branch must be present locally before `git worktree add` can
# reach it.
!git fetch --quiet origin '+refs/heads/*:refs/remotes/origin/*'
!git branch -r

In [ ]:
# 2. Dependencies
!pip install -q -r pybuildingenergy/requirements.txt
!pip install -q plotly            # interactive Sankey; the PNG works without it

# git needs an identity before `worktree add` will run in a fresh container
!git config user.email "colab@example.com"
!git config user.name  "Colab"
print("dependencies installed")

In [ ]:
# 3. Pin the weather file — ONE file for every run in this notebook.
#
# The EPW ships in the repo, so nothing is downloaded and no run can quietly
# fall back to a different source. EnergyPlus can only read an EPW anyway.
import glob, sys
sys.path.insert(0, "examples")

EPW = sorted(glob.glob("weather_cache/*.epw"))[0]

from weather_melbourne import read_epw_site, site_offset_deg
lat, lon, city = read_epw_site(EPW)
print(f"weather file : {EPW}")
print(f"station      : {city}  (lat {lat}, lon {lon})")
print(f"offset from central Melbourne: {site_offset_deg(lat, lon):.2f} deg")

# Expected: Melbourne.RO at lat -37.8075, lon 144.97, offset 0.01 deg.
# Anything near 2 deg is the old inland Charlton file — re-clone.

# To use a different station, upload it and re-run this cell:
#   from google.colab import files; files.upload()
#   !mv *.epw weather_cache/
# Sources: https://climate.onebuilding.org
#   -> WMO Region 5 > AUS_Australia > VIC_Victoria

---

# Step 1 — Unmodified pyBuildingEnergy vs EnergyPlus

Same building, same weather, same schedules, same setpoints — the **unmodified**
ISO 52016-1 engine against EnergyPlus 24.1.

The alignment audit in cell 6 is the point of this step, not a preamble. Several
ISO behaviours are invisible in the building dictionary, and three of them
dominate the comparison: internal gains ignore the dictionary's `full_load`
values, neighbours are ISO 13789 *unconditioned buffers* rather than rooms held
at 21 °C, and control is on **operative** temperature, not air temperature.
Pinning the neighbours at a fixed 21 °C in a naive IDF changes total energy by
~40×.

In [ ]:
# 4. Install EnergyPlus 24.1  (~190 MB, a minute or two)
EP_URL = ("https://github.com/NREL/EnergyPlus/releases/download/v24.1.0/"
          "EnergyPlus-24.1.0-9d7789a3ac-Linux-Ubuntu22.04-x86_64.tar.gz")
!wget -q -O /tmp/ep.tar.gz $EP_URL
!mkdir -p /opt/ep && tar -xzf /tmp/ep.tar.gz -C /opt/ep --strip-components=1
!/opt/ep/energyplus --version

In [ ]:
# 5. Parameter alignment audit — prints the table and writes the IDF, runs nothing.
#    22 parameters checked: 10 already aligned, 9 corrected, 3 irreducible.
!python examples/baseline_vs_energyplus.py --audit-only

In [ ]:
# 6. Both engines, on the pinned EPW.
#    --require-epw is implied here: EnergyPlus cannot read a PVGIS TMY.
!python examples/baseline_vs_energyplus.py \
    --energyplus /opt/ep/energyplus \
    --weather "$EPW" \
    --outdir results/baseline_vs_ep

In [ ]:
# 7. Step 1 result — table, bar chart, and the pyBuildingEnergy Sankey.
import pandas as pd
from IPython.display import Image, display, Markdown

display(Markdown("### Heating, cooling and total — ISO 52016-1 vs EnergyPlus"))
display(pd.read_csv("results/baseline_vs_ep/baseline_vs_energyplus.csv"))
display(Image("results/baseline_vs_ep/baseline_vs_energyplus.png"))

display(Markdown("### pyBuildingEnergy annual energy balance (Sankey)"))
display(Image("results/baseline_vs_ep/sankey_pybuildingenergy.png"))

The Sankey balance does **not** close exactly, and the gap is drawn rather than
hidden. ISO 52016-1 is a node network, not one lumped air node: gains are split
between the air node and the surface nodes, which then exchange with each other,
so the air-node paths summed here need not add up. On this EPW the residual is
about 10 %.

In [ ]:
# 8. Interactive Sankey (optional).
#
# Rendered from a figure object, NOT from the .html file on disk. Pointing an
# IFrame at a local path is what produces "localhost refused to connect" —
# Colab serves no web server on that path. The PNG above always works; this is
# the hoverable version.
import json, sys
sys.path.insert(0, "examples")
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

from baseline_vs_energyplus import sankey_flows, SANKEY_IN_COLORS, SANKEY_OUT_COLORS

iso = json.load(open("results/baseline_vs_ep/iso_results.json"))
inflows, outflows = sankey_flows(iso)

labels = [n for n, _ in inflows] + ["Zone"] + [n for n, _ in outflows]
z = len(inflows)
colors = ([SANKEY_IN_COLORS[i] for i in range(len(inflows))] + ["#52514e"]
          + [SANKEY_OUT_COLORS[i] for i in range(len(outflows))])

fig = go.Figure(go.Sankey(
    node=dict(pad=18, thickness=18, label=labels, color=colors),
    link=dict(
        source=list(range(z)) + [z] * len(outflows),
        target=[z] * z + list(range(z + 1, len(labels))),
        value=[v for _, v in inflows] + [v for _, v in outflows],
        color=[colors[i] for i in range(z)]
              + [colors[z + 1 + i] for i in range(len(outflows))],
    ),
))
fig.update_layout(title="Apt 305 — ISO 52016-1 annual energy balance (kWh/yr)",
                  height=560)
fig.show()

---

# Steps 2 and 3 — the window changes, added one at a time

Three engines on the **same** building and the **same** EPW, so the engine is the
only thing that varies:

| Column | Branch | Engine |
| --- | --- | --- |
| **Baseline** | `claude/pybuildingenergy-baseline-anjro8` | unmodified ISO 52016-1 |
| **+ Window** | `claude/dynamic-window-properties-anjro8` | + dynamic window properties |
| **+ Window + h_ce** | `claude/window-plus-dynamic-hce-anjro8` | …+ wind-dependent thermal transmittance on every surface |

Each branch runs in its own subprocess out of a throwaway git worktree. That
isolation is not optional: three versions of the same `pybuildingenergy` package
cannot coexist on one `sys.path`, and the second import would silently resolve to
the first.

> **Read the middle column carefully.** `+ Window` is *not* the solar-angle
> correction alone — that branch changes two things at once: the angular
> correction factor `F_W(θ)` on transmitted solar **and** a wind-dependent
> thermal transmittance on the windows. `+ Window + h_ce` then extends that same
> wind-dependent transmittance to the opaque envelope. See the note under the
> results table for how to split the middle column if you want the angle effect
> on its own.

In [ ]:
# 9. Three engine branches on the pinned EPW (slow cell — three annual simulations)
!python examples/compare_branches_apt305.py \
    --weather "$EPW" --require-epw --outdir results/apt305

In [ ]:
# 10. Steps 2–3 result — table and chart.
#     "Base" = the engine with no change at all, and it is bit-identical to the
#     ISO column of step 1 (asserted in the last cell).
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/apt305/comparison.csv"))
display(Image("results/apt305/apt305_comparison.png"))

### Splitting the middle column

Both parts of the window change are switchable, so the angle effect can be
isolated without leaving the top branch. Run the engine with
`window_angular_solar_model="none"` to keep only the dynamic transmittance, or
with `window_convection_model="table"` to keep only the angular correction —
that second one is the true *window angle alone* column.

---

# Step 4 — back to the original engine, plus ventilation and latent heat

Four engines, starting again from the **unmodified** baseline:

| Column | Branch | Engine |
| --- | --- | --- |
| **Base** | `claude/pybuildingenergy-baseline-anjro8` | unmodified ISO 52016-1 |
| **C1** | `claude/ventilation-infiltration-fix` | + ventilation / infiltration fix |
| **C2** | `claude/latent-heat-fix` | + latent heat fix |
| **C3** | `claude/ventilation-plus-latent-fix` | both together |

These are a **separate** layering from steps 2–3 — they do not stack on the
window branches, which is why the run starts from the baseline again.

In [ ]:
# 11. Four engine branches on the pinned EPW (slow cell — four annual simulations)
!python examples/compare_ventilation_latent.py \
    --weather "$EPW" --require-epw --outdir results/ventilation_latent

In [ ]:
# 12. Step 4 result — table and chart.
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("results/ventilation_latent/comparison.csv"))
display(Image("results/ventilation_latent/ventilation_latent_comparison.png"))

Two of those metrics — *Latent heating need* and *Total energy need* — are
derived in the comparison script rather than read from the engine. Upstream
exposes latent heating only as an hourly column and never sums the three
demands; adding them to the annual aggregation would mean editing the baseline,
which has to stay byte-identical to upstream. They are computed from columns that
exist on all four branches, so every variant is measured by the same definition.

---

## The cross-check

Steps 1 and 2–3 both drive the *unmodified* engine on the same building, so their
baseline figures must be identical. This cell compares the **weather each run
actually used** (from `run_meta.json`) before comparing any numbers, so a weather
mismatch is reported as a weather mismatch and not as a physics bug.

In [ ]:
# 13. THE CHECK: both harnesses must report the same baseline engine result.
!python examples/check_baseline_consistency.py